# 🍎 HUẤN LUYỆN YOLOv8 NHẬN DIỆN & PHÂN LOẠI 100 LOẠI TRÁI CÂY (ĐỀ TÀI 22)
## ⚡ HUẤN LUYỆN SIÊU TỐC TRÊN GOOGLE COLAB VỚI GPU T4 MIỄN PHÍ
---
> **Hướng dẫn chuẩn bị trước khi chạy:**
> 1. Vào menu **Thời gian chạy (Runtime)** -> **Thay đổi loại thời gian chạy (Change runtime type)** -> Chọn **T4 GPU** -> Bấm **Lưu (Save)**.
> 2. Tải file `fruit_dataset_yolo.zip` được đóng gói từ Web App ở máy tính của bạn lên thư mục Colab này.

In [ ]:
# BƯỚC 1: KIỂM TRA GPU T4 TRÊN GOOGLE COLAB
!nvidia-smi

In [ ]:
# BƯỚC 2: CÀI ĐẶT THƯ VIỆN ULTRALYTICS YOLOV8 CHÍNH THỨC
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

In [ ]:
# BƯỚC 3: GIẢI NÉN FILE DỮ LIỆU ĐÃ ĐÓNG GÓI TỪ WEB APP
import os, zipfile

zip_filename = 'fruit_dataset_yolo.zip'
if os.path.exists(zip_filename):
    print(f'[*] Đang giải nén {zip_filename}...')
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall('.')
    print('[OK] Giải nén dữ liệu hoàn tất!')
    print('Các thư mục trong dataset:', os.listdir('dataset') if os.path.exists('dataset') else 'Chưa thấy thư mục dataset!')
else:
    print(f'[!] Hãy bấm nút biểu tượng tệp ở thanh bên trái Colab và tải file {zip_filename} lên trước khi chạy bước này!')

In [ ]:
# BƯỚC 4: HUẤN LUYỆN MÔ HÌNH YOLOV8 CLASSIFICATION SIÊU TỐC TRÊN GPU
from ultralytics import YOLO

# Khởi tạo mô hình cơ sở YOLOv8 Nano Classifier
model = YOLO('yolov8n-cls.pt')

# Tiến hành huấn luyện tự động với 25 Epochs (Khoảng 2 - 4 phút trên T4 GPU)
results = model.train(
    data='dataset',
    epochs=25,
    imgsz=224,
    batch=32,
    device=0,
    project='fruit_yolo',
    name='fruit_model',
    exist_ok=True
)

print('[OK] Huấn luyện hoàn thành xuất sắc!')

In [ ]:
# BƯỚC 5: XUẤT THÊM ĐỊNH DẠNG ONNX (ĐỂ CHẠY SIÊU NHẸ MỌI NƠI)
import shutil

best_pt = 'fruit_yolo/fruit_model/weights/best.pt'
if os.path.exists(best_pt):
    trained_model = YOLO(best_pt)
    # Xuất ra file best.onnx
    trained_model.export(format='onnx')
    print('[OK] Đã xuất file mô hình ONNX thành công!')

In [ ]:
# BƯỚC 6: XEM BIỂU ĐỒ KẾT QUẢ HUẤN LUYỆN & TẢI MÔ HÌNH VỀ MÁY
from IPython.display import Image, display
from google.colab import files

# Hiển thị biểu đồ huấn luyện Loss và Accuracy
results_chart = 'fruit_yolo/fruit_model/results.png'
if os.path.exists(results_chart):
    print('Biểu đồ huấn luyện:')
    display(Image(results_chart))

# Tự động tải file best.onnx (Khuyên dùng - tương thích 100% Windows) và best.pt về máy tính
best_onnx = 'fruit_yolo/fruit_model/weights/best.onnx'
if os.path.exists(best_onnx):
    print('[*] Đang tải file best.onnx về máy tính của bạn...')
    files.download(best_onnx)
if os.path.exists(best_pt):
    print('[*] Đang tải file best.pt về máy tính của bạn...')
    files.download(best_pt)
print('[THÀNH CÔNG] Hãy kéo thả file best.onnx hoặc best.pt vào Web App (Tab 4) để kích hoạt ngay!')